In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv(r"Z:\Data\AL2011CN_publicdata_YR-removed.csv")

In [3]:
df.columns

Index(['SEQN', 'OPASCST2', 'OPXDGRBL', 'OPXSGRBL', 'OPDURET', 'OPDURL4',
       'OPDUME', 'OPDUARM', 'OPDUBVO', 'OPDUCVO', 'OPDDVCDR', 'OPXDRCDR',
       'OPDSVCDR', 'OPXSRCDR', 'RIAGENDR', 'RIDAGEYR', 'RIDRETH1', 'LBXGH',
       'DIQ010', 'DIQ050'],
      dtype='object')

In [4]:
#NHANES respondents
df['SEQN'].nunique()

6797

In [5]:
#subset df to only columns that will be used for analysis
df2 = df[['SEQN', 'OPXDGRBL', 'OPXSGRBL', 'OPDURET', 'OPDUME', 'OPDUARM', 'OPDUBVO', 'OPDUCVO', 'OPDDVCDR', 'OPXDRCDR',
       'OPDSVCDR', 'OPXSRCDR', 'RIAGENDR', 'RIDAGEYR', 'RIDRETH1', 'LBXGH', 'DIQ010', 'DIQ050']]

In [6]:
#drop rows with null values
df3 = df2.dropna(subset=['OPXDGRBL', 'OPXSGRBL', 'OPDURET', 'OPDUME', 'OPDUARM', 'OPDUBVO', 'OPDUCVO', 'OPDDVCDR', 'OPXDRCDR',
       'OPDSVCDR', 'OPXSRCDR'], how = 'all')

In [7]:
df3['SEQN'].nunique()

5828

In [8]:
### Age
df3['RIDAGEYR'].describe()

count    5828.000000
mean       59.915065
std        12.546225
min        40.000000
25%        49.000000
50%        60.000000
75%        70.000000
max        85.000000
Name: RIDAGEYR, dtype: float64

In [9]:
#Gender
df3['RIAGENDR'].value_counts()

1    2924
2    2904
Name: RIAGENDR, dtype: int64

In [10]:
#Race/ethnicity
df3['RIDRETH1'].value_counts()

3    3138
4    1202
1     894
2     407
5     187
Name: RIDRETH1, dtype: int64

#### Recode retinopathy level as follows:
- Levels 10-13 = No retinopathy (0)
- Levels 14-20 = Mild NPR (1)
- Levels 31-41 = Moderate NPR (2)
- Level 51 = Severe NPR (3)
- Levels 60-80 = PR (Proliferative) (4)

In [11]:
#Retinopathy level in worse eye
df3['OPDURET'].value_counts()

10.0    4611
20.0     297
13.0     270
15.0     131
31.0     119
12.0     113
41.0      94
14.0      26
51.0      12
65.0      10
63.0       7
60.0       7
64.0       3
61.0       2
70.0       2
Name: OPDURET, dtype: int64

In [12]:
df3['RetLev'] = pd.cut(x=df['OPDURET'],
                     bins = [0, 13, 20, 41, 51, 80],
                     labels = [0, 1, 2, 3, 4])

In [13]:
df3['RetLev'].value_counts()

0    4994
1     454
2     213
4      31
3      12
Name: RetLev, dtype: int64

#### Add in ungradeable element

In [14]:
df3['OPXDGRBL'].value_counts()

0.0    5300
2.0     261
3.0     123
6.0      37
5.0      22
1.0      14
4.0       9
Name: OPXDGRBL, dtype: int64

In [15]:
df3['OPXSGRBL'].value_counts()

0.0    5296
2.0     245
3.0     133
6.0      37
5.0      23
1.0      19
4.0      13
Name: OPXSGRBL, dtype: int64

In [16]:
#check if the ungradables are same across eyes
od = df3.loc[df3['OPXDGRBL']==6]
os = df3.loc[df3['OPXSGRBL']==6]

In [17]:
od_ungr = od['SEQN'].unique()
os_ungr = os['SEQN'].unique()
a = set(od_ungr)
print(len(a))
b = set(os_ungr)
print(len(b))

37
37


In [18]:
if a==b:
    print("lists are equal")
else:
    print("lists aren't equal")

lists aren't equal


#### Check gradability OD, OS, Both:
    - B = ungradeable both eyes
    - OD = only OD ungradeable
    - OS = only OS ungradeable

In [19]:
def categorise(df3):
    if df3['OPXDGRBL'] ==6 and df3['OPXSGRBL'] == 6:
        return 'B'
    elif df3['OPXDGRBL'] == 6:
        return 'OD_only'
    elif df3['OPXSGRBL'] == 6:
        return 'OS_only'

In [20]:
df3['Ungrade'] = df3.apply(lambda df3: categorise(df3), axis = 1)

In [21]:
#how many people were ungradeable in both eyes (B)
df3['Ungrade'].value_counts()

OD_only    26
OS_only    26
B          11
Name: Ungrade, dtype: int64

#### In csv changed any rows where RetLev was empty but at least one eye was ungradable to 5


In [22]:
df3.to_csv(r"Z:\Output\Data tables\EyeFields.csv")

In [24]:
df3 = pd.read_csv(r"Z:\Output\Data tables\EyeFields2.csv")

In [25]:
df3['RetLev'].value_counts()

0.0    4994
1.0     454
2.0     213
5.0      39
4.0      31
3.0      12
Name: RetLev, dtype: int64

#### Recode for Diabetes present
- 1 = Yes: 
    - LBXGH defined as A1c>=6.5% or 
    - DIQ010 yes to having been told by doctor or
    - DIQ050 taking insulin
- 2 = No

In [26]:
df3['LBXGH'].describe()

count    5639.000000
mean        5.855169
std         1.102716
min         2.000000
25%         5.300000
50%         5.600000
75%         6.000000
max        15.200000
Name: LBXGH, dtype: float64

In [27]:
df3['DIQ010'].value_counts()

2    4775
1     919
3     127
9       7
Name: DIQ010, dtype: int64

In [28]:
df3['DIQ050'].value_counts()

2    5600
1     228
Name: DIQ050, dtype: int64

In [29]:
def categorise(row):
    if row['LBXGH'] >=6.5 or row['DIQ010'] == 1 or row['DIQ050'] == 1:
        return 1
    return 2

In [30]:
df3['Diabetes'] = df3.apply(lambda row: categorise(row), axis = 1)

In [31]:
#Number of respondents with diabetes (=1)
df3['Diabetes'].value_counts()

2    4680
1    1148
Name: Diabetes, dtype: int64

#### Table 3 Macular Edema OPDUME
- present (1) = NHANES 2|3
- absent (0) = NHANES 0|1

In [32]:
df3['OPDUME'].value_counts()

0.0    5587
3.0      47
1.0      29
2.0      21
7.0       7
Name: OPDUME, dtype: int64

In [33]:
def categorise(df3):
    if df3['OPDUME'] <= 1:
        return 0
    elif df3['OPDUME'] == 2:
        return 1
    elif df3['OPDUME'] == 3:
        return 1
    elif df3['OPDUME'] == 7:
        return 7

In [34]:
df3['DME'] = df3.apply(lambda df3: categorise(df3), axis = 1)

In [35]:
df3['DME'].value_counts()

0.0    5616
1.0      68
7.0       7
Name: DME, dtype: int64

#### Table 4 AMD

In [36]:
#rename AMD field
df3['AMD'] = df3['OPDUARM']

In [37]:
df3['AMD'].value_counts()

0.0    5163
1.0     386
2.0      55
Name: AMD, dtype: int64

#### Table 5 RVO
- Absent (0) = NHANES OPDUBVO and OPDUCVO 0|1
- Present (1) = NHANES OPDUBVO and OPDUCVO 2|3
*lists are not the same so grade as present if respondent had presence in either field

In [38]:
df3['OPDUBVO'].value_counts()

0.0    5716
2.0      29
1.0      19
Name: OPDUBVO, dtype: int64

In [39]:
df3['OPDUCVO'].value_counts()

0.0    5763
2.0       5
1.0       4
3.0       1
Name: OPDUCVO, dtype: int64

In [40]:
def categorise(df3):
    if df3['OPDUBVO'] >= 2 or df3['OPDUCVO'] >= 2:
        return 1
    else: 
        return 0

In [41]:
df3['RVO'] = df3.apply(lambda df3: categorise(df3), axis = 1)

In [42]:
#how many respondents have RVO present in either eye
df3['RVO'].value_counts()

0    5793
1      35
Name: RVO, dtype: int64

#### Table 6 First Glaucoma & Table 7 Re-Read Glaucoma
*continuos var and has laterality
NHANES vars = OPDDVCDR + OPDSVCDR | OPXDRCDR + OPXSRCDR

- VCTDR < 0.6 = 1
- VCTDR >= 0.6 = 2

In [43]:
df3['OPDDVCDR'].describe()

count    5603.000000
mean        0.411662
std         0.119787
min         0.080000
25%         0.320000
50%         0.410000
75%         0.490000
max         0.920000
Name: OPDDVCDR, dtype: float64

In [44]:
df3['OPDSVCDR'].describe()

count    5603.000000
mean        0.408801
std         0.119543
min         0.120000
25%         0.320000
50%         0.400000
75%         0.490000
max         0.900000
Name: OPDSVCDR, dtype: float64

In [45]:
def categorise(df3):
    if df3['OPDDVCDR'] < 0.6 or df3['OPDSVCDR'] < 0.6 :
        return 1
    elif df3['OPDDVCDR'] >= 0.6 or df3['OPDSVCDR'] >= 0.6 :
        return 2

In [46]:
df3['VCDR'] = df3.apply(lambda df3: categorise(df3), axis = 1)

In [47]:
df3['VCDR'].value_counts()

1.0    5519
2.0     227
Name: VCDR, dtype: int64

In [48]:
df3['OPXDRCDR'].describe()

count    720.000000
mean       0.577569
std        0.272436
min        0.000000
25%        0.400000
50%        0.600000
75%        0.700000
max        2.000000
Name: OPXDRCDR, dtype: float64

In [49]:
df3['OPXDRCDR'].describe()

count    720.000000
mean       0.577569
std        0.272436
min        0.000000
25%        0.400000
50%        0.600000
75%        0.700000
max        2.000000
Name: OPXDRCDR, dtype: float64

In [50]:
def categorise(df3):
    if df3['OPXDRCDR'] < 0.6 or df3['OPXDRCDR'] < 0.6 :
        return 1
    elif df3['OPXDRCDR'] >= 0.6 or df3['OPXDRCDR'] >= 0.6 :
        return 2

In [51]:
df3['RRVCDR'] = df3.apply(lambda df3: categorise(df3), axis = 1)

In [52]:
df3['RRVCDR'].value_counts()

2.0    418
1.0    302
Name: RRVCDR, dtype: int64

#### Output file for Tables 2 through 7
#### SEQN | DR | ME | AMD | RVO | VCDR | RRVCDR |
---------------------------------------------------------------------

In [53]:
df4 = df3[['SEQN', 'Ungrade', 'RetLev', 'DME', 'AMD', 'RVO','VCDR','RRVCDR']]

In [54]:
#are there any completely empty rows
df4.isna().all(1).sum()

0

In [55]:
df4['SEQN'].nunique()

5828

In [56]:
df4.to_csv(r"Z:\Output\Data tables\EyeFields3.csv")